# Graphs for paper

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from math import pi
from matplotlib.lines import Line2D

In [2]:
## below is code for some of the graphs in the paper

In [13]:
plt.rcParams.update({
    "font.size": 15, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

df = pd.read_csv("results_data_from_tables.csv")
datasets = ["MyoSegmenTUM", "Pathological", "AIPS", "Sheffield", "Augmented"]
order = (df.groupby("algorithm")["dice"].mean().sort_values(ascending=False).index.tolist())

# Per-dataset colors + hatching (grayscale-safe) for the bar chart
ds_colors = {"MyoSegmenTUM": "#0072B2", "Pathological": "#D55E00",
             "AIPS": "#009E73", "Sheffield": "#CC79A7", "Augmented": "#E69F00"}
ds_hatch = {"MyoSegmenTUM": "", "Pathological": "//", "AIPS": "..",
            "Sheffield": "xx", "Augmented": "\\\\"}

# Per-algorithm color + linestyle + marker (grayscale-safe) for the radar
algo_style = {
    "MuscleMap WB":           dict(color="#0072B2", ls="-",  marker="o"),
    "MuscleMap Thigh":        dict(color="#56B4E9", ls="--", marker="s"),
    "Multimodal-Multiethnic": dict(color="#CC79A7", ls="-.", marker="^"),
    "MuSeg":                  dict(color="#009E73", ls=":",  marker="D"),
    "MedCLIP-SAMv2":          dict(color="#E69F00", ls="--", marker="v"),
    "Dafne":                  dict(color="#D55E00", ls="-.", marker="P"),
    #"MedSegDiff":             dict(color="#555555", ls=":",  marker="X"),
}

In [14]:
# ================= Dice grouped bar chart (no reference line) =================
fig, ax = plt.subplots(figsize=(13, 6.5))
n_ds = len(datasets); group_w = 0.82; bar_w = group_w / n_ds
x = np.arange(len(order))
for i, ds in enumerate(datasets):
    vals = [df[(df.algorithm == a) & (df.dataset == ds)]["dice"].values[0] for a in order]
    offsets = x - group_w/2 + bar_w*(i + 0.5)
    ax.bar(offsets, vals, bar_w, label=ds, color=ds_colors[ds],
           hatch=ds_hatch[ds], edgecolor="white", linewidth=0.5)
ax.set_xticks(x); ax.set_xticklabels(order, rotation=25, ha="right")
ax.set_ylabel("Dice coefficient"); ax.set_ylim(0, 1)
ax.legend(title="Dataset", ncol=5, loc="upper center",
          bbox_to_anchor=(0.5, -0.22), frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("dice_bars.png", dpi=200, bbox_inches="tight")
plt.savefig("dice_bars.pdf", bbox_inches="tight")
plt.close()
print("saved dice_bars")

saved dice_bars


In [15]:
## here insert dice bars with confidence intrvals

In [16]:
print(type(order))
print(order)

<class 'list'>
['MuscleMap WB', 'MuscleMap Thigh', 'Multimodal-Multiethnic', 'MedCLIP-SAMv2', 'MuSeg', 'Dafne', 'MedSegDiff']


In [17]:
# here drop medsegdiff from order

del order[6]
print(order)

['MuscleMap WB', 'MuscleMap Thigh', 'Multimodal-Multiethnic', 'MedCLIP-SAMv2', 'MuSeg', 'Dafne']


In [18]:
# ================= Dice bars with 95% CI (real per-subject data) =================
# Reconstructed from raw per-case CSVs at ../<algo>/codes/results_<dataset>/df_*.csv
# Only AIPS, Sheffield, and Augmented have surviving per-subject raw data on disk;
# MyoSegmenTUM/Pathological raw case files were deleted after the summary means were
# computed, so those two datasets are excluded here (no fabricated CIs).
# AIPS ground truth only labels the right side, so "_L"/"_left" columns are dropped
# for AIPS only (they are 0.0 dice against an empty mask, not real failures).
import glob, os
from scipy import stats

ci_datasets = ["AIPS", "Sheffield", "Augmented"]

raw_folders = {
    "MuscleMap WB":           {"AIPS": "../muscle_map_wb/codes/results_asian_water",
                                "Sheffield": "../muscle_map_wb/codes/results_sheffield",
                                "Augmented": "../muscle_map_wb/codes/results_augmented"},
    "MuscleMap Thigh":        {"AIPS": "../muscle_map_thigh/codes/results_asian_water",
                                "Sheffield": "../muscle_map_thigh/codes/results_sheffield",
                                "Augmented": "../muscle_map_thigh/codes/results_augmented"},
    "Multimodal-Multiethnic": {"AIPS": "../multimodal-multiethnic/codes/results_asian_water",
                                "Sheffield": "../multimodal-multiethnic/codes/results_sheffield",
                                "Augmented": "../multimodal-multiethnic/codes/results_augmented"},
    "MuSeg":                  {"AIPS": "../museg/codes/results_asian_water_only",
                                "Sheffield": "../museg/codes/results_sheffield",
                                "Augmented": "../museg/codes/results_augmented"},
    "MedCLIP-SAMv2":          {"AIPS": "../medclipsamv2textboxes/codes/results_asian_water",
                                "Sheffield": "../medclipsamv2textboxes/codes/results_sheffield",
                                "Augmented": "../medclipsamv2textboxes/augmented_results"},
    "Dafne":                  {"AIPS": "../dafne/codes/results_asian_water",
                                "Sheffield": "../dafne/codes/results_sheffield",
                                "Augmented": "../dafne/codes/results_augmented"},

}

def per_subject_dice(folder, drop_left):
    files = glob.glob(os.path.join(folder, "df_*.csv"))
    merged = None
    for f in files:
        d = pd.read_csv(f)
        id_col = d.columns[0]
        dice_cols = [c for c in d.columns if c.endswith("_dice")]
        if drop_left:
            dice_cols = [c for c in dice_cols if not c[:-5].endswith(("_L", "_left"))]
        if not dice_cols:
            continue
        sub = d[[id_col] + dice_cols].set_index(id_col)
        merged = sub if merged is None else merged.join(sub, how="outer")
    return merged.mean(axis=1).dropna()

ci_stats = {}
for algo in order:
    for ds in ci_datasets:
        vals = per_subject_dice(raw_folders[algo][ds], drop_left=(ds == "AIPS"))
        n = len(vals)
        mean = vals.mean()
        sem = vals.std(ddof=1) / np.sqrt(n)
        halfwidth = stats.t.ppf(0.975, n - 1) * sem
        ci_stats[(algo, ds)] = (mean, halfwidth, n)
        print(f"{algo:26s} {ds:10s} n={n:3d} mean={mean:.3f} +/- {halfwidth:.3f}")

# ---- grouped bar chart with 95% CI error bars ----
fig, ax = plt.subplots(figsize=(11, 6.5))
n_ds = len(ci_datasets); group_w = 0.7; bar_w = group_w / n_ds
x = np.arange(len(order))
for i, ds in enumerate(ci_datasets):
    vals = [ci_stats[(a, ds)][0] for a in order]
    errs = [ci_stats[(a, ds)][1] for a in order]
    offsets = x - group_w/2 + bar_w*(i + 0.5)
    ax.bar(offsets, vals, bar_w, yerr=errs, capsize=3, label=ds, color=ds_colors[ds],
           hatch=ds_hatch[ds], edgecolor="white", linewidth=0.5,
           error_kw=dict(ecolor="black", elinewidth=1))
ax.set_xticks(x); ax.set_xticklabels(order, rotation=25, ha="right")
ax.set_ylabel("Dice coefficient"); ax.set_ylim(0, 1)
ax.legend(title="Dataset", ncol=3, loc="upper center",
          bbox_to_anchor=(0.5, -0.22), frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("dice_bars_ci.png", dpi=200, bbox_inches="tight")
plt.savefig("dice_bars_ci.pdf", bbox_inches="tight")
plt.close()
print("saved dice_bars_ci")

MuscleMap WB               AIPS       n= 25 mean=0.861 +/- 0.013
MuscleMap WB               Sheffield  n= 38 mean=0.697 +/- 0.018
MuscleMap WB               Augmented  n= 20 mean=0.796 +/- 0.037
MuscleMap Thigh            AIPS       n= 25 mean=0.850 +/- 0.013
MuscleMap Thigh            Sheffield  n= 69 mean=0.687 +/- 0.013
MuscleMap Thigh            Augmented  n= 20 mean=0.693 +/- 0.072
Multimodal-Multiethnic     AIPS       n= 25 mean=0.574 +/- 0.016
Multimodal-Multiethnic     Sheffield  n= 69 mean=0.587 +/- 0.014
Multimodal-Multiethnic     Augmented  n= 20 mean=0.558 +/- 0.101
MedCLIP-SAMv2              AIPS       n= 25 mean=0.456 +/- 0.018
MedCLIP-SAMv2              Sheffield  n= 69 mean=0.619 +/- 0.015
MedCLIP-SAMv2              Augmented  n= 20 mean=0.134 +/- 0.038
MuSeg                      AIPS       n= 25 mean=0.663 +/- 0.069
MuSeg                      Sheffield  n= 69 mean=0.566 +/- 0.020
MuSeg                      Augmented  n= 20 mean=0.315 +/- 0.130
Dafne                    

In [20]:
# ================= 4-metric radar grid (larger text) =================
N = len(datasets)
angles = [n/float(N)*2*pi for n in range(N)]; angles += angles[:1]
metrics = [("dice", "Dice", False), ("jaccard", "Jaccard", False),
           ("bound_iou", "Boundary IoU", False), ("hausdorff", "Hausdorff (inverted)", True)]

fig, axes = plt.subplots(2, 2, figsize=(15, 15.5), subplot_kw=dict(polar=True))
axes = axes.flatten()
for ax, (col, title, invert) in zip(axes, metrics):
    vmin, vmax = df[col].min(), df[col].max()
    for algo in order:
        st = algo_style[algo]
        raw = [df[(df.algorithm==algo)&(df.dataset==d)][col].values[0] for d in datasets]
        norm = [(1-(v-vmin)/(vmax-vmin)) if invert else (v-vmin)/(vmax-vmin) for v in raw]
        norm += norm[:1]
        ax.plot(angles, norm, linewidth=2.2, color=st["color"], linestyle=st["ls"],
                marker=st["marker"], markersize=6)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(datasets, fontsize=15)
    ax.set_ylim(0, 1); ax.set_yticklabels([])
    ax.set_title(title, fontsize=19, fontweight="bold", pad=28)
    ax.tick_params(axis='x', pad=10)
handles = [Line2D([0],[0], color=algo_style[a]["color"], ls=algo_style[a]["ls"],
                  marker=algo_style[a]["marker"], markersize=9, linewidth=2.2) for a in order]
fig.legend(handles, order, loc="upper center", ncol=4, frameon=False,
           fontsize=17, bbox_to_anchor=(0.5, 1.04))
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("multimetric_radar.png", dpi=170, bbox_inches="tight")
plt.savefig("multimetric_radar.pdf", bbox_inches="tight")
plt.close()
print("saved multimetric_radar")

saved multimetric_radar


In [21]:
# ================= 4-metric radar grid (dice/jaccard/bound_iou unscaled, raw 0-1) =================
# Dice, Jaccard, and Boundary IoU are already bounded in [0, 1], so they're plotted as-is
# instead of being min/max normalized. Hausdorff is unbounded, so it keeps the
# invert + min/max-normalize treatment (needed to put it on the same 0-1 radial axis).
N = len(datasets)
angles = [n/float(N)*2*pi for n in range(N)]; angles += angles[:1]
metrics = [("dice", "Dice", False), ("jaccard", "Jaccard", False),
           ("bound_iou", "Boundary IoU", False), ("hausdorff", "Hausdorff (inverted)", True)]

fig, axes = plt.subplots(2, 2, figsize=(15, 15.5), subplot_kw=dict(polar=True))
axes = axes.flatten()
for ax, (col, title, invert) in zip(axes, metrics):
    for algo in order:
        st = algo_style[algo]
        raw = [df[(df.algorithm==algo)&(df.dataset==d)][col].values[0] for d in datasets]
        if invert:
            vmin, vmax = df[col].min(), df[col].max()
            vals = [1-(v-vmin)/(vmax-vmin) for v in raw]
        else:
            vals = raw
        vals = list(vals) + [vals[0]]
        ax.plot(angles, vals, linewidth=2.2, color=st["color"], linestyle=st["ls"],
                marker=st["marker"], markersize=6)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(datasets, fontsize=15)
    ax.set_ylim(0, 1); ax.set_yticklabels([])
    ax.set_title(title, fontsize=19, fontweight="bold", pad=28)
    ax.tick_params(axis='x', pad=10)
handles = [Line2D([0],[0], color=algo_style[a]["color"], ls=algo_style[a]["ls"],
                  marker=algo_style[a]["marker"], markersize=9, linewidth=2.2) for a in order]
fig.legend(handles, order, loc="upper center", ncol=4, frameon=False,
           fontsize=17, bbox_to_anchor=(0.5, 1.04))
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("multimetric_radar_unscaled.png", dpi=170, bbox_inches="tight")
plt.savefig("multimetric_radar_unscaled.pdf", bbox_inches="tight")
plt.close()
print("saved multimetric_radar_unscaled")

saved multimetric_radar_unscaled
